<a href="https://colab.research.google.com/github/vanessabfabri/-Criando-um-Dashboard-de-Vendas-do-Xbox-com-Excel/blob/main/Criando_um_Dashboard_de_Vendas_do_Xbox_com_Excel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


### Criando um Dashboard de Vendas do Xbox com Excel
Exercício da Plataforma DIO: Criar um dashboard de vendas no Excel.

In [18]:
from google.colab import files

uploaded = files.upload()


Saving 0120950e-64c8-4092-a257-ba22ed198c69.xlsx to 0120950e-64c8-4092-a257-ba22ed198c69.xlsx


In [20]:
# 1. Instalação e Importação
!pip install -q xlsxwriter
import pandas as pd
import os
from datetime import datetime

# 2. Localização Automática do Arquivo
# Busca qualquer arquivo que comece com '0120950e' para evitar erros de digitação
diretorio = '/content/'
arquivos = [f for f in os.listdir(diretorio) if f.startswith('0120950e')]

if not arquivos:
    print("❌ ERRO: Arquivo não encontrado! Certifique-se de que fez o upload do .xlsx na pasta lateral.")
else:
    file_path = os.path.join(diretorio, arquivos[0])
    print(f"✅ Arquivo detectado: {file_path}")

    # 3. Carregamento e Limpeza de Dados (ETL)
    # Lendo a aba de Bases do Excel
    df = pd.read_excel(file_path, sheet_name='B̳ases')

    df['Start Date'] = pd.to_datetime(df['Start Date'])

    def clean_money(val):
        if isinstance(val, str):
            val = val.replace('-', '0').replace('$', '').strip()
        try:
            return float(val)
        except:
            return 0.0

    df['Total Value'] = df['Total Value'].apply(clean_money)

    # 4. Cálculos para o Dashboard
    total_rev = df['Total Value'].sum()
    total_subs = df['Subscriber ID'].nunique()
    avg_ticket = total_rev / total_subs if total_subs > 0 else 0

    rev_plan = df.groupby('Plan')['Total Value'].sum().sort_values(ascending=False).reset_index()
    rev_cycle = df.groupby('Subscription Type')['Total Value'].sum().reset_index()

    # 5. Geração do Excel Premium com XlsxWriter
    output_name = 'Dashboard_Xbox_Premium_Final.xlsx'
    writer = pd.ExcelWriter(output_name, engine='xlsxwriter')

    # Exporta base bruta
    df.to_excel(writer, sheet_name='Dados_Processados', index=False)

    workbook = writer.book
    ws = workbook.add_worksheet('Dashboard')
    ws.hide_gridlines(2) # Visual limpo sem grades

    # --- DEFINIÇÃO DE ESTILOS PROFISSIONAIS ---
    fmt_title = workbook.add_format({'bold': True, 'font_size': 26, 'font_color': 'white', 'bg_color': '#107C10', 'align': 'center', 'valign': 'vcenter'})
    fmt_kpi_lbl = workbook.add_format({'bold': True, 'font_size': 11, 'font_color': '#595959', 'align': 'center', 'bg_color': '#F2F2F2', 'border': 1, 'border_color': '#D9D9D9'})
    fmt_kpi_val = workbook.add_format({'bold': True, 'font_size': 22, 'font_color': '#107C10', 'align': 'center', 'bg_color': '#F2F2F2', 'border': 1, 'border_color': '#D9D9D9'})
    fmt_header = workbook.add_format({'bold': True, 'bg_color': '#107C10', 'font_color': 'white', 'border': 1, 'align': 'center'})
    fmt_money = workbook.add_format({'num_format': '$#,##0.00', 'border': 1})
    fmt_border = workbook.add_format({'border': 1})

    # --- MONTAGEM DO LAYOUT ---
    ws.set_column('A:A', 2)
    ws.set_column('B:M', 16)

    # Cabeçalho Principal
    ws.merge_range('B2:M3', 'XBOX GAME PASS: PERFORMANCE EXECUTIVE DASHBOARD', fmt_title)
    ws.write('M4', f"Atualizado em: {datetime.now().strftime('%d/%m/%Y')}", workbook.add_format({'italic': True, 'align': 'right', 'font_size': 9}))

    # Seção de KPIs (Cartões)
    kpi_list = [("RECEITA TOTAL", f"${total_rev:,.2f}"), ("ASSINANTES", f"{total_subs}"), ("TICKET MÉDIO", f"${avg_ticket:,.2f}")]
    col_pos = [1, 5, 9] # Colunas B, F, J
    for i, (label, val) in enumerate(kpi_list):
        pos = col_pos[i]
        ws.merge_range(5, pos, 5, pos + 2, label, fmt_kpi_lbl)
        ws.merge_range(6, pos, 7, pos + 2, val, fmt_kpi_val)

    # Tabelas de Suporte (Invisíveis sob os gráficos)
    ws.write('B10', 'PLANO', fmt_header); ws.write('C10', 'RECEITA', fmt_header)
    for i, row in rev_plan.iterrows():
        ws.write(11 + i, 1, row['Plan'], fmt_border)
        ws.write(11 + i, 2, row['Total Value'], fmt_money)

    # --- INSERÇÃO DE GRÁFICOS ---
    # Gráfico 1: Receita por Plano
    chart1 = workbook.add_chart({'type': 'column'})
    chart1.add_series({
        'categories': ['Dashboard', 11, 1, 10 + len(rev_plan), 1],
        'values':     ['Dashboard', 11, 2, 10 + len(rev_plan), 2],
        'fill':       {'color': '#107C10'},
        'data_labels': {'value': True}
    })
    chart1.set_title({'name': 'Faturamento por Categoria de Plano'})
    chart1.set_legend({'none': True})
    ws.insert_chart('B15', chart1, {'x_scale': 1.2, 'y_scale': 1})

    # Gráfico 2: Share de Ciclo (Pizza)
    # Escreve dados do ciclo para o gráfico
    ws.write('H10', 'CICLO', fmt_header); ws.write('I10', 'VALOR', fmt_header)
    for i, row in rev_cycle.iterrows():
        ws.write(11 + i, 7, row['Subscription Type'], fmt_border)
        ws.write(11 + i, 8, row['Total Value'], fmt_money)

    chart2 = workbook.add_chart({'type': 'pie'})
    chart2.add_series({
        'categories': ['Dashboard', 11, 7, 10 + len(rev_cycle), 7],
        'values':     ['Dashboard', 11, 8, 10 + len(rev_cycle), 8],
        'data_labels': {'percentage': True, 'position': 'outside_end'}
    })
    chart2.set_title({'name': 'Participação por Ciclo de Cobrança'})
    ws.insert_chart('H15', chart2, {'x_scale': 1.2, 'y_scale': 1})

    writer.close()
    print(f"✅ Dashboard Premium gerado com sucesso! Arquivo: {output_name}")

✅ Arquivo detectado: /content/0120950e-64c8-4092-a257-ba22ed198c69.xlsx
✅ Dashboard Premium gerado com sucesso! Arquivo: Dashboard_Xbox_Premium_Final.xlsx
